# CrowdSetu CSRNet GPU Training
This notebook is a reproducible Colab/Kaggle launcher. It does not contain measured results until it is run against a real ShanghaiTech dataset.

Provide a dataset mounted at `DATASET_ROOT` with `train_data` and `test_data` folders, or update the path in the configuration cell.

In [ ]:
# Install the non-Torch dependencies in the notebook runtime.
# Colab/Kaggle supplies the CUDA-compatible Torch build; do not replace it with
# the Windows CPU pin used for local smoke tests.
!pip install -q -r requirements-gpu.txt
!pip install -q pytest

In [ ]:
from pathlib import Path
import torch

print('torch.cuda.is_available()', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a CUDA runtime before training'
DATASET_ROOT = Path('/content/ShanghaiTech/part_B_final')
OUTPUT_DIR = Path('/content/crowdsetu_checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert (DATASET_ROOT / 'train_data/images').is_dir(), f'Missing dataset: {DATASET_ROOT}'
print('torch', torch.__version__)
print('cuda_device', torch.cuda.get_device_name(0))
print(DATASET_ROOT, OUTPUT_DIR)

In [ ]:
# Generate density maps for both splits before training.
!python generate_density_maps.py --dataset_root {DATASET_ROOT} --split train_data --sigma_mode fixed --sigma 15
!python generate_density_maps.py --dataset_root {DATASET_ROOT} --split test_data --sigma_mode fixed --sigma 15
!python scripts/verify_density_maps.py --dataset_root {DATASET_ROOT} --split train_data
!python scripts/verify_density_maps.py --dataset_root {DATASET_ROOT} --split test_data

In [ ]:
# Train with CUDA AMP, resumable checkpoints, CSV history, TensorBoard, and metrics JSON.
!python train_csrnet.py \
  --train_images {DATASET_ROOT}/train_data/images \
  --train_density {DATASET_ROOT}/train_data/density_maps \
  --test_images {DATASET_ROOT}/test_data/images \
  --test_density {DATASET_ROOT}/test_data/density_maps \
  --device cuda --amp \
  --epochs 100 --batch_size 4 --seed 42 --validation_fraction 0.15 \
  --checkpoint_dir {OUTPUT_DIR} \
  --csv_path {OUTPUT_DIR}/training_history.csv \
  --tensorboard_dir {OUTPUT_DIR}/tensorboard \
  --metrics_path {OUTPUT_DIR}/metrics.json

In [ ]:
# If the runtime disconnects, resume from the last full checkpoint.
!python train_csrnet.py \
  --train_images {DATASET_ROOT}/train_data/images \
  --train_density {DATASET_ROOT}/train_data/density_maps \
  --test_images {DATASET_ROOT}/test_data/images \
  --test_density {DATASET_ROOT}/test_data/density_maps \
  --device cuda --amp --resume {OUTPUT_DIR}/last_checkpoint.pt \
  --checkpoint_dir {OUTPUT_DIR} \
  --metrics_path {OUTPUT_DIR}/metrics.json

## Download outputs
Download `csrnet_best.pth`, `last_checkpoint.pt`, `training_history.csv`, `metrics.json`, and the `tensorboard/` directory from `OUTPUT_DIR`. Do not describe the metrics as project results until the run has been completed and the hardware, dataset split, and command are recorded.